# Extraction & Summarization Evaluation

Alright, this is where we find out if our pipeline actually *works*.

Building an extraction and summarization system is one thing. Knowing
whether it's any good is another thing entirely. You wouldn't ship a
circuit board without testing it, and you shouldn't ship an NLP pipeline
without measuring its output quality.

Here's what we're doing in this notebook:

1. **Entity extraction comparison**: Run the SpaCy baseline and (optionally)
   GCP NL API on the same documents and compare what they find
2. **Summarization quality assessment**: Generate Gemini summaries of CNN/DailyMail
   articles, then compute ROUGE scores against the human-written gold-standard
   highlights
3. **Qualitative spot-checks**: Eyeball the best, worst, and median summaries
   to understand what the numbers mean in practice

The CNN/DailyMail dataset is our secret weapon here — it comes with human-written
summaries, which means we have a known-good reference signal to calibrate against.
That's rare and valuable.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import textwrap
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from src.data.loader import load_news, load_reviews, load_support_tickets
from src.data.preprocessing import preprocess_documents
from src.extraction.spacy_baseline import SpacyExtractor
from src.summarization.evaluation import SummarizationEvaluator

print('All imports loaded. Let\'s measure some quality!')

## 1. Entity Extraction — SpaCy Baseline

First, let's run SpaCy's NER across all three source types and see what
kinds of entities it finds. This establishes our baseline — the quality
bar that the GCP NL API needs to clear to justify its cost.

We're using `en_core_web_sm` (the small model) because it's fast and
gives us a fair representation of what a free, local NER engine can do.

In [ ]:
# Load a manageable sample from each source
reviews = load_reviews(max_docs=200)
tickets = load_support_tickets(max_docs=200)
news = load_news(max_docs=100)

# Preprocess
all_docs = reviews + tickets + news
all_docs = preprocess_documents(all_docs)

print(f'Loaded {len(reviews)} reviews, {len(tickets)} tickets, {len(news)} news articles')

In [ ]:
# Fire up SpaCy
spacy_extractor = SpacyExtractor(model_name='en_core_web_sm')

# Extract entities from each source type
extraction_results = {}
for source_name, docs in [('reviews', reviews), ('tickets', tickets), ('news', news)]:
    results = spacy_extractor.extract_batch([d.text for d in docs[:100]])
    extraction_results[source_name] = results
    
    # Count entity types
    all_labels = [e.label for r in results for e in r.entities]
    label_counts = Counter(all_labels)
    total_entities = sum(len(r.entities) for r in results)
    avg_per_doc = total_entities / max(len(results), 1)
    
    print(f'\n{source_name.upper()}: {total_entities} entities ({avg_per_doc:.1f} avg/doc)')
    for label, count in label_counts.most_common(8):
        print(f'  {label:15s}: {count:4d}')

In [ ]:
# Visualize entity type distributions across sources
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (source_name, results) in enumerate(extraction_results.items()):
    all_labels = [e.label for r in results for e in r.entities]
    label_counts = Counter(all_labels).most_common(10)
    
    if label_counts:
        labels, counts = zip(*label_counts)
        colors = ['#4285F4', '#EA4335', '#34A853']
        axes[i].barh(list(labels), list(counts), color=colors[i])
        axes[i].set_title(f'Entity Types: {source_name}')
        axes[i].invert_yaxis()

plt.tight_layout()
plt.show()

### Extraction Quality — Qualitative Check

Let's look at a few actual extraction results to understand what
SpaCy is finding (and missing). Numbers are good, but eyeballs are
essential.

In [ ]:
# Spot-check a few extractions from each source
for source_name, docs, results in [
    ('REVIEW', reviews, extraction_results['reviews']),
    ('TICKET', tickets, extraction_results['tickets']),
    ('NEWS', news, extraction_results['news']),
]:
    print(f'\n{"=" * 70}')
    print(f' {source_name} — Sample Extraction')
    print(f'{"=" * 70}')
    
    # Pick a doc with a decent number of entities
    for j, (doc, result) in enumerate(zip(docs, results)):
        if len(result.entities) >= 3:
            print(f'\nText: {textwrap.shorten(doc.text, 200)}')
            print(f'Entities found: {len(result.entities)}')
            for ent in result.entities[:8]:
                print(f'  [{ent.label:12s}] {ent.text}')
            if result.noun_chunks:
                print(f'Key noun chunks: {", ".join(result.noun_chunks[:5])}')
            break

## 2. Summarization Quality — ROUGE Evaluation

This is the headline evaluation. We'll generate summaries for CNN/DailyMail
articles and measure them against the human-written highlights using ROUGE.

**If you have GCP credentials** and Vertex AI access, the next cell will call
Gemini to generate real summaries. **If not**, we'll use a simple extractive
baseline (first 2-3 sentences) to demonstrate the evaluation framework —
the measurement machinery works the same either way.

### What are ROUGE scores?

ROUGE measures how much overlap there is between a generated summary and
a reference summary:
- **ROUGE-1**: Unigram (single word) overlap — are the right words present?
- **ROUGE-2**: Bigram (word pair) overlap — are the right phrases present?
- **ROUGE-L**: Longest common subsequence — does the overall flow match?

Higher is better. State-of-the-art models typically score 0.40-0.50 on
ROUGE-1 F1 for news summarization.

In [ ]:
# Attempt Gemini summarization; fall back to extractive baseline
USE_GEMINI = False  # Set to True if you have GCP Vertex AI credentials

if USE_GEMINI:
    from src.summarization.vertex_summarize import GeminiSummarizer
    PROJECT_ID = 'your-gcp-project-id'  # <-- UPDATE THIS
    summarizer = GeminiSummarizer(project_id=PROJECT_ID)
    print('Using Gemini via Vertex AI for summarization')
else:
    # Extractive baseline: take the first 3 sentences as the "summary".
    # This is intentionally naive — it shows what you get without any
    # intelligence, and gives us a lower bound for comparison.
    from nltk.tokenize import sent_tokenize
    print('Using extractive baseline (first 3 sentences) — set USE_GEMINI=True for Vertex AI')

In [ ]:
# Generate summaries for our news articles
# The news docs have gold-standard highlights in their metadata
eval_sample = news[:50]  # 50 articles is a solid evaluation set

generated_summaries = []
reference_summaries = []
source_texts = []
doc_ids = []

for doc in eval_sample:
    reference = doc.metadata.get('highlights', '')
    if not reference:
        continue
    
    if USE_GEMINI:
        generated = summarizer.summarize(doc.text)
    else:
        # Extractive baseline: first 3 sentences
        sentences = sent_tokenize(doc.text)
        generated = ' '.join(sentences[:3])
    
    generated_summaries.append(generated)
    reference_summaries.append(reference)
    source_texts.append(doc.text)
    doc_ids.append(doc.id)

print(f'Generated {len(generated_summaries)} summaries for evaluation')

In [ ]:
# Score everything with ROUGE
evaluator = SummarizationEvaluator()

results = evaluator.score_batch(
    generated_summaries=generated_summaries,
    reference_summaries=reference_summaries,
    doc_ids=doc_ids,
    source_texts=source_texts,
)

# Aggregate and print the report
aggregate = evaluator.aggregate_scores(results)
method_name = 'Gemini (Vertex AI)' if USE_GEMINI else 'Extractive Baseline (first 3 sentences)'
evaluator.print_report(aggregate, title=f'Summarization Evaluation — {method_name}')

In [ ]:
# Visualize ROUGE score distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, metric_name in enumerate(['rouge1', 'rouge2', 'rougeL']):
    f1_scores = [getattr(r.rouge_scores, f'{metric_name.replace("-", "")}_f1') for r in results]
    # Handle the naming: rouge1_f1, rouge2_f1, rougeL_f1
    if metric_name == 'rouge1':
        f1_scores = [r.rouge_scores.rouge1_f1 for r in results]
    elif metric_name == 'rouge2':
        f1_scores = [r.rouge_scores.rouge2_f1 for r in results]
    else:
        f1_scores = [r.rouge_scores.rougeL_f1 for r in results]
    
    axes[i].hist(f1_scores, bins=20, color='#4285F4', alpha=0.7, edgecolor='white')
    axes[i].axvline(np.mean(f1_scores), color='#EA4335', linestyle='--', label=f'Mean: {np.mean(f1_scores):.3f}')
    axes[i].set_title(f'{metric_name.upper()} F1 Distribution')
    axes[i].set_xlabel('F1 Score')
    axes[i].set_ylabel('Count')
    axes[i].legend()

plt.suptitle(f'ROUGE Score Distributions — {method_name}', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Qualitative Spot-Checks

Numbers tell you the average, but you learn the most from looking at
the extremes. Let's read the best, worst, and median summaries side
by side with their references. This is where you develop intuition
about what the model gets right and where it stumbles.

In [ ]:
spot_checks = evaluator.qualitative_spot_check(results)

for check in spot_checks:
    print(f'\n{"=" * 70}')
    print(f' {check["label"]} (ROUGE-1 F1: {check["rouge1_f1"]:.4f})')
    print(f' Document: {check["doc_id"]}')
    print(f'{"=" * 70}')
    print(f'\n  GENERATED:')
    print(textwrap.fill(check['generated'], width=70, initial_indent='    ', subsequent_indent='    '))
    print(f'\n  REFERENCE (human-written):')
    print(textwrap.fill(check['reference'], width=70, initial_indent='    ', subsequent_indent='    '))

## 4. Length Analysis

Are our summaries the right length? Too short and they miss information.
Too long and they're not really summaries. We compare generated lengths
against reference lengths and source lengths.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

gen_lens = [r.generated_length for r in results]
ref_lens = [r.reference_length for r in results]

# Generated vs Reference lengths
axes[0].scatter(ref_lens, gen_lens, alpha=0.5, color='#4285F4')
max_len = max(max(gen_lens), max(ref_lens))
axes[0].plot([0, max_len], [0, max_len], 'r--', alpha=0.5, label='Perfect match')
axes[0].set_xlabel('Reference Length (words)')
axes[0].set_ylabel('Generated Length (words)')
axes[0].set_title('Generated vs Reference Summary Length')
axes[0].legend()

# Compression ratios
compressions = [r.compression_ratio for r in results if r.compression_ratio > 0]
axes[1].hist(compressions, bins=20, color='#34A853', alpha=0.7, edgecolor='white')
axes[1].axvline(np.mean(compressions), color='#EA4335', linestyle='--',
               label=f'Mean: {np.mean(compressions):.3f}')
axes[1].set_title('Compression Ratio (summary / source)')
axes[1].set_xlabel('Ratio')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Key Findings & Next Steps

### What we measured
- **Entity extraction**: SpaCy baseline establishes entity type distributions
  across all three source types. News articles are entity-dense (people, orgs,
  locations). Reviews focus on products and quantities. Tickets are sparser.
  Running the GCP NL API (with `USE_GEMINI=True`) lets you compare managed
  service quality against this baseline.

- **Summarization quality**: ROUGE scores give us quantitative quality measures.
  The extractive baseline sets the floor — any intelligent summarizer should
  beat first-3-sentences. Gemini (when enabled) should show significant
  improvements, especially on ROUGE-2 and ROUGE-L which capture phrase-level
  and structural quality.

### What to improve next
- **Prompt tuning**: The summarization prompt can be optimized per source type.
  News articles need different treatment than product reviews.
- **Structured extraction evaluation**: Build a similar scoring framework for
  Gemini's structured extraction (core issues, action items, topics).
- **Cross-source comparison**: Run the same evaluation framework across all
  three source types to see where the pipeline performs best and worst.
- **Cost tracking**: Measure API call costs alongside quality to find the
  optimal quality/cost trade-off for production.

The evaluation framework is now in place. Every pipeline change from here
can be measured against this baseline. That's how you make real progress —
measure, change, measure again.